# Session 5 — Model Context Protocol (MCP) Fundamentals

## Focus: Standardised tool & memory interfaces with MCP

### Introduction to Model Context Protocol (MCP)

The Model Context Protocol (MCP) is a standardized framework that enables large language models to interact with external tools, resources, and data sources in a consistent and secure manner. Think of MCP as a universal adapter that allows LLMs to connect with various external services and capabilities, much like how USB provides a standard way for devices to connect to computers.

MCP addresses a critical challenge in AI development: how to safely and efficiently extend LLM capabilities beyond their training data without requiring model retraining or fine-tuning. By providing a standardized interface, MCP enables developers to create reusable tools that can work across different LLM platforms and applications.

The protocol operates on a client-server architecture where the LLM application acts as a client that can discover and utilize tools exposed by MCP servers. This separation of concerns allows for better security, scalability, and maintainability compared to baking tool capabilities directly into the LLM.

In [ ]:
# Install required packages for MCP
!pip install mcp langchain langchain-openai python-dotenv

# Key Concepts Deep Dive

### MCP Server Architecture and Communication Protocols

**MCP Server Components:**
An MCP server consists of several key components that work together to provide tools and resources to LLM clients:

1. **Tool Registry**: A catalog of available functions that the server can execute
2. **Resource Manager**: Handles access to external data sources and content
3. **Execution Engine**: Processes tool calls and manages their execution
4. **Transport Layer**: Manages communication with clients

**Transport Protocols:**
MCP supports two primary communication protocols, each with distinct advantages:

1. **stdio (Standard Input/Output)**:
   - **Operation**: Uses traditional stdin/stdout streams for communication
   - **Best For**: Local development, debugging, and simple integrations
   - **Advantages**: Simple to implement, no network configuration needed
   - **Limitations**: Limited to local execution, no remote access capability

2. **HTTP-SSE (HTTP with Server-Sent Events)**:
   - **Operation**: Uses HTTP requests for initial setup and SSE for real-time communication
   - **Best For**: Production deployments, remote access, scalable applications
   - **Advantages**: Supports remote connections, better error handling, scalable
   - **Limitations**: More complex implementation, requires network configuration

**Communication Flow:**
The typical MCP communication follows this pattern:
1. **Handshake**: Client and server establish connection and negotiate capabilities
2. **Discovery**: Client requests list of available tools and resources
3. **Execution**: Client calls specific tools with parameters
4. **Response**: Server returns results or errors
5. **Cleanup**: Connection is properly terminated

**Security Considerations:**
- **Authentication**: Servers can implement authentication mechanisms
- **Authorization**: Fine-grained control over which tools can be accessed
- **Input Validation**: All inputs are validated before processing
- **Rate Limiting**: Protection against excessive requests
- **Execution Sandboxing**: Tools run in isolated environments when possible

### MCP Tool Architecture and Design Patterns

**Tool Definition Structure:**
Each MCP tool follows a standardized schema that includes:

1. **Name**: A unique identifier using snake_case convention
2. **Description**: Clear explanation of the tool's purpose and functionality
3. **Parameters**: Defined using JSON Schema specification
4. **Return Type**: Expected output format and structure
5. **Error Handling**: Standardized error responses and codes

**Parameter Design:**
MCP tools use JSON Schema for parameter definition, which includes:
- **Data Types**: string, number, boolean, array, object
- **Validation Rules**: min/max values, patterns, required fields
- **Default Values**: Predefined values for optional parameters
- **Enumerations**: Restricted sets of allowed values
- **Nested Structures**: Complex parameter objects with sub-properties

**Tool Categories:**
MCP tools can be categorized based on their functionality:

1. **Information Retrieval Tools**:
   - Fetch data from databases, APIs, or file systems
   - Examples: database queries, web API calls, file readers

2. **Calculation Tools**:
   - Perform mathematical or logical operations
   - Examples: calculators, unit converters, data transformers

3. **Action Execution Tools**:
   - Perform actions in external systems
   - Examples: sending emails, updating databases, controlling devices

4. **Resource Access Tools**:
   - Read from or write to external resources
   - Examples: file editors, database managers, cloud storage

**Implementation Patterns:**

1. **Stateless Tools**:
   - Each call is independent and doesn't rely on previous executions
   - Easier to implement and scale
   - Example: Calculator, unit converter

2. **Stateful Tools**:
   - Maintain state between calls
   - Require session management
   - Example: Shopping cart, multi-step wizard

3. **Composite Tools**:
   - Combine multiple operations into a single tool
   - Reduce round-trip communications
   - Example: Weather service that returns current and forecast data

4. **Streaming Tools**:
   - Handle large or continuous data streams
   - Use chunked responses or Server-Sent Events
   - Example: Real-time data feeds, file uploads/downloads

**Error Handling Strategy:**
MCP implements robust error handling through:
- **Validation Errors**: Invalid parameters or malformed requests
- **Execution Errors**: Failures during tool execution
- **Resource Errors**: Missing or inaccessible resources
- **Network Errors**: Communication failures
- **Timeout Errors**: Operations taking too long

Each error type includes:
- Standardized error codes
- Human-readable messages
- Diagnostic information
- Recovery suggestions

**Performance Considerations:**
- **Caching**: Implement caching for frequently accessed data
- **Batching**: Combine multiple operations when possible
- **Async Processing**: Handle long-running operations asynchronously
- **Rate Limiting**: Protect against abuse and ensure fair usage
- **Monitoring**: Track performance metrics and usage patterns

**Security Best Practices:**
- **Input Sanitization**: Validate and clean all inputs
- **Access Control**: Implement proper authentication and authorization
- **Resource Limits**: Set boundaries on memory, CPU, and time usage
- **Audit Logging**: Record all tool executions for security analysis
- **Secure Defaults**: Ensure safe configuration out of the box

## Hands-On: Spin up mcp-sandbox, expose calculator & weather tools

In [5]:
# First, let's set up a simple MCP server with calculator and weather tools
import asyncio
from typing import Any, Dict, List
import json

# Create a simple MCP server implementation
class CalculatorWeatherServer:
    def __init__(self):
        self.name = "calculator-weather-server"

    async def list_tools(self):
        """Return available tools"""
        return [
            {
                "name": "calculator",
                "description": "Perform mathematical calculations",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "Mathematical expression to evaluate"
                        }
                    },
                    "required": ["expression"]
                }
            },
            {
                "name": "weather",
                "description": "Get weather information for a location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "City name or coordinates"
                        },
                        "units": {
                            "type": "string",
                            "description": "Temperature units (celsius or fahrenheit)",
                            "enum": ["celsius", "fahrenheit"],
                            "default": "celsius"
                        }
                    },
                    "required": ["location"]
                }
            }
        ]

    async def call_tool(self, name, arguments):
        """Execute a tool"""
        if name == "calculator":
            return await self._calculate(arguments)
        elif name == "weather":
            return await self._get_weather(arguments)
        else:
            return f"Error: Unknown tool: {name}"

    async def _calculate(self, arguments):
        """Calculate mathematical expression"""
        import math

        expression = arguments.get("expression", "")

        # Safe evaluation
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return "Error: Expression contains invalid characters"

        try:
            result = eval(expression, {"__builtins__": None}, {"math": math})
            return json.dumps({"result": result, "status": "success"})
        except Exception as e:
            return json.dumps({"error": str(e), "status": "error"})

    async def _get_weather(self, arguments):
        """Get weather information (simulated)"""
        location = arguments.get("location", "")
        units = arguments.get("units", "celsius")

        # Simulated weather data
        weather_data = {
            "new york": {"celsius": 22, "fahrenheit": 72, "condition": "Partly Cloudy"},
            "london": {"celsius": 15, "fahrenheit": 59, "condition": "Rainy"},
            "tokyo": {"celsius": 28, "fahrenheit": 82, "condition": "Sunny"},
            "sydney": {"celsius": 18, "fahrenheit": 64, "condition": "Cloudy"},
            "paris": {"celsius": 20, "fahrenheit": 68, "condition": "Clear"}
        }

        location_key = location.lower()
        if location_key in weather_data:
            temp = weather_data[location_key][units]
            condition = weather_data[location_key]["condition"]
            return json.dumps({
                "location": location,
                "temperature": temp,
                "units": units,
                "condition": condition,
                "status": "success"
            })
        else:
            return json.dumps({
                "error": f"Weather data not available for {location}",
                "status": "error"
            })

# Create and test the server
server = CalculatorWeatherServer()

# Use a different approach to run async code in Jupyter
async def test_server():
    print("Testing CalculatorWeatherServer:")
    print("=" * 40)

    # Test listing tools
    tools = await server.list_tools()
    print("Available tools:")
    for tool in tools:
        print(f"  - {tool['name']}: {tool['description']}")

    # Test calling tools
    print("\nTesting calculator:")
    result1 = await server.call_tool("calculator", {"expression": "2 + 3 * 4"})
    print(f"Result: {result1}")

    print("\nTesting weather:")
    result2 = await server.call_tool("weather", {"location": "London", "units": "celsius"})
    print(f"Result: {result2}")

    print("\nTesting unknown tool:")
    result3 = await server.call_tool("unknown", {})
    print(f"Result: {result3}")

# Run the test using a different approach for Jupyter
try:
    # Try to get the current event loop
    loop = asyncio.get_event_loop()
except RuntimeError:
    # If there's no current event loop, create a new one
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

# Run the async function
if loop.is_running():
    # If the loop is already running, create a task
    task = loop.create_task(test_server())
else:
    # If the loop is not running, run it until complete
    loop.run_until_complete(test_server())

Testing CalculatorWeatherServer:
Available tools:
  - calculator: Perform mathematical calculations
  - weather: Get weather information for a location

Testing calculator:
Result: {"result": 14, "status": "success"}

Testing weather:
Result: {"location": "London", "temperature": 15, "units": "celsius", "condition": "Rainy", "status": "success"}

Testing unknown tool:
Result: Error: Unknown tool: unknown


## Wrapping MCP Tools in LangChain

In [8]:
# Synchronous MCP server implementation
import json
import math

class SyncCalculatorWeatherServer:
    def __init__(self):
        self.name = "sync-calculator-weather-server"

    def list_tools(self):
        """Return available tools"""
        return [
            {
                "name": "calculator",
                "description": "Perform mathematical calculations",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "Mathematical expression to evaluate"
                        }
                    },
                    "required": ["expression"]
                }
            },
            {
                "name": "weather",
                "description": "Get weather information for a location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "City name or coordinates"
                        },
                        "units": {
                            "type": "string",
                            "description": "Temperature units (celsius or fahrenheit)",
                            "enum": ["celsius", "fahrenheit"],
                            "default": "celsius"
                        }
                    },
                    "required": ["location"]
                }
            }
        ]

    def call_tool(self, name, arguments):
        """Execute a tool"""
        if name == "calculator":
            return self._calculate(arguments)
        elif name == "weather":
            return self._get_weather(arguments)
        else:
            return json.dumps({
                "error": f"Unknown tool: {name}",
                "status": "error"
            })

    def _calculate(self, arguments):
        """Calculate mathematical expression"""
        expression = arguments.get("expression", "")

        # Safe evaluation
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return json.dumps({
                "error": "Expression contains invalid characters",
                "status": "error"
            })

        try:
            result = eval(expression, {"__builtins__": None}, {"math": math})
            return json.dumps({
                "result": result,
                "status": "success"
            })
        except Exception as e:
            return json.dumps({
                "error": str(e),
                "status": "error"
            })

    def _get_weather(self, arguments):
        """Get weather information (simulated)"""
        location = arguments.get("location", "")
        units = arguments.get("units", "celsius")

        # Simulated weather data
        weather_data = {
            "new york": {"celsius": 22, "fahrenheit": 72, "condition": "Partly Cloudy"},
            "london": {"celsius": 15, "fahrenheit": 59, "condition": "Rainy"},
            "tokyo": {"celsius": 28, "fahrenheit": 82, "condition": "Sunny"},
            "sydney": {"celsius": 18, "fahrenheit": 64, "condition": "Cloudy"},
            "paris": {"celsius": 20, "fahrenheit": 68, "condition": "Clear"}
        }

        location_key = location.lower()
        if location_key in weather_data:
            temp = weather_data[location_key][units]
            condition = weather_data[location_key]["condition"]
            return json.dumps({
                "location": location,
                "temperature": temp,
                "units": units,
                "condition": condition,
                "status": "success"
            })
        else:
            return json.dumps({
                "error": f"Weather data not available for {location}",
                "status": "error"
            })

# Create the synchronous server instance
sync_server = SyncCalculatorWeatherServer()

# Test the server directly
print("Testing Sync Server Directly:")
print("=" * 40)

result1 = sync_server.call_tool("calculator", {"expression": "2 + 3 * 4"})
print(f"Calculator result: {result1}")

result2 = sync_server.call_tool("weather", {"location": "London", "units": "celsius"})
print(f"Weather result: {result2}")

result3 = sync_server.call_tool("unknown", {})
print(f"Unknown tool result: {result3}")

Testing Sync Server Directly:
Calculator result: {"result": 14, "status": "success"}
Weather result: {"location": "London", "temperature": 15, "units": "celsius", "condition": "Rainy", "status": "success"}
Unknown tool result: {"error": "Unknown tool: unknown", "status": "error"}


### Integrating MCP with LangChain Framework

**Integration Architecture:**
LangChain provides a robust framework for integrating MCP tools through several key components:

1. **Tool Wrappers**: Bridge between MCP tool definitions and LangChain tool interface
2. **Parameter Mapping**: Convert between LangChain's parameter format and MCP's JSON Schema
3. **Error Handling**: Translate MCP errors into LangChain-compatible exceptions
4. **Session Management**: Handle connection pooling and session persistence

**Wrapper Implementation Patterns:**

1. **Direct Wrapper Pattern**:
   - Simple 1:1 mapping between MCP tools and LangChain tools
   - Minimal abstraction layer
   - Best for simple tools with straightforward parameters

2. **Composite Wrapper Pattern**:
   - Combine multiple MCP tools into a single LangChain tool
   - Handle complex workflows internally
   - Better user experience for common task combinations

3. **Adaptive Wrapper Pattern**:
   - Dynamically adjust based on available MCP tools
   - Fallback mechanisms for unavailable tools
   - Automatic tool discovery and registration

**Parameter Handling:**
LangChain uses Pydantic models for parameter validation, which aligns well with MCP's JSON Schema approach:

1. **Type Mapping**: Convert between JSON Schema types and Python types
2. **Validation**: Apply both MCP and LangChain validation rules
3. **Default Values**: Handle optional parameters with default values
4. **Complex Types**: Support for nested objects and arrays

**Error Handling Integration:**
- **Exception Translation**: Convert MCP error responses into LangChain exceptions
- **Retry Logic**: Implement automatic retries for transient errors
- **Circuit Breakers**: Prevent cascading failures in distributed systems
- **Fallback Behavior**: Provide alternative approaches when tools fail

**Performance Optimization:**
- **Connection Pooling**: Reuse MCP server connections across tool calls
- **Caching Layer**: Cache tool definitions and frequently used results
- **Batch Processing**: Combine multiple tool calls when possible
- **Lazy Loading**: Load tool definitions only when needed

**Security Integration:**
- **Credential Management**: Secure handling of authentication tokens
- **Access Control**: Enforce role-based access to MCP tools
- **Input Validation**: Multi-layer validation (LangChain + MCP)
- **Audit Logging**: Comprehensive logging of all tool executions

**Advanced Integration Features:**

1. **Tool Discovery**: Automatic detection and registration of available MCP tools
2. **Dynamic Tool Loading**: Load tools on-demand based on application needs
3. **Tool Composition**: Create new tools by combining existing MCP tools
4. **Quality of Service**: Prioritize tool execution based on importance
5. **Monitoring Integration**: Connect with observability platforms for monitoring

**Best Practices for LangChain Integration:**

1. **Abstraction Level**: Choose the right level of abstraction for your use case
2. **Error Handling**: Implement comprehensive error handling at multiple levels
3. **Performance**: Consider latency and throughput requirements
4. **Security**: Follow security best practices for both LangChain and MCP
5. **Testing**: Thoroughly test both successful and error scenarios
6. **Documentation**: Provide clear documentation for integrated tools
7. **Versioning**: Handle version compatibility between LangChain and MCP
8. **Monitoring**: Implement comprehensive monitoring and alerting

**Common Integration Challenges:**

1. **Type System Mismatches**: Differences between JSON Schema and Python types
2. **Error Handling Consistency**: Ensuring consistent error handling across layers
3. **Performance Overhead**: Minimizing latency introduced by the integration layer
4. **Security Compliance**: Meeting security requirements across both systems
5. **Version Compatibility**: Managing updates to both LangChain and MCP
6. **Debugging Complexity**: Troubleshooting across multiple layers of abstraction

**Future Evolution:**
The integration between LangChain and MCP continues to evolve with:
- Tighter native integration in future LangChain versions
- Improved tool discovery and registration mechanisms
- Enhanced performance optimization features
- Better security and compliance capabilities
- More sophisticated error handling and recovery patterns

In [15]:
# First, ensure we have all necessary imports
import json
import math
import os
from langchain.tools import Tool
from langchain.agents import AgentType, initialize_agent
from langchain_openai import ChatOpenAI
from langchain.schema import SystemMessage
from langchain.memory import ConversationBufferMemory

# Synchronous MCP server implementation
class SyncCalculatorWeatherServer:
    def __init__(self):
        self.name = "sync-calculator-weather-server"

    def list_tools(self):
        """Return available tools"""
        return [
            {
                "name": "calculator",
                "description": "Perform mathematical calculations",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "Mathematical expression to evaluate"
                        }
                    },
                    "required": ["expression"]
                }
            },
            {
                "name": "weather",
                "description": "Get weather information for a location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "City name or coordinates"
                        },
                        "units": {
                            "type": "string",
                            "description": "Temperature units (celsius or fahrenheit)",
                            "enum": ["celsius", "fahrenheit"],
                            "default": "celsius"
                        }
                    },
                    "required": ["location"]
                }
            }
        ]

    def call_tool(self, name, arguments):
        """Execute a tool"""
        if name == "calculator":
            return self._calculate(arguments)
        elif name == "weather":
            return self._get_weather(arguments)
        else:
            return json.dumps({
                "error": f"Unknown tool: {name}",
                "status": "error"
            })

    def _calculate(self, arguments):
        """Calculate mathematical expression"""
        expression = arguments.get("expression", "")

        # Safe evaluation
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return json.dumps({
                "error": "Expression contains invalid characters",
                "status": "error"
            })

        try:
            result = eval(expression, {"__builtins__": None}, {"math": math})
            return json.dumps({
                "result": result,
                "status": "success"
            })
        except Exception as e:
            return json.dumps({
                "error": str(e),
                "status": "error"
            })

    def _get_weather(self, arguments):
        """Get weather information (simulated)"""
        location = arguments.get("location", "")
        units = arguments.get("units", "celsius")

        # Simulated weather data
        weather_data = {
            "new york": {"celsius": 22, "fahrenheit": 72, "condition": "Partly Cloudy"},
            "london": {"celsius": 15, "fahrenheit": 59, "condition": "Rainy"},
            "tokyo": {"celsius": 28, "fahrenheit": 82, "condition": "Sunny"},
            "sydney": {"celsius": 18, "fahrenheit": 64, "condition": "Cloudy"},
            "paris": {"celsius": 20, "fahrenheit": 68, "condition": "Clear"}
        }

        location_key = location.lower()
        if location_key in weather_data:
            temp = weather_data[location_key][units]
            condition = weather_data[location_key]["condition"]
            return json.dumps({
                "location": location,
                "temperature": temp,
                "units": units,
                "condition": condition,
                "status": "success"
            })
        else:
            return json.dumps({
                "error": f"Weather data not available for {location}",
                "status": "error"
            })

# Create the synchronous server instance
sync_server = SyncCalculatorWeatherServer()

# Create LangChain tools from our MCP server functions
def calculator_tool_function(expression: str) -> str:
    """Wrapper for the calculator tool"""
    try:
        result = sync_server.call_tool("calculator", {"expression": expression})
        result_data = json.loads(result)
        if result_data.get("status") == "success":
            return f"Calculation result: {result_data['result']}"
        else:
            return f"Error: {result_data.get('error', 'Unknown error')}"
    except Exception as e:
        return f"Tool execution error: {str(e)}"

def weather_tool_function(input_str: str) -> str:
    """Wrapper for the weather tool that parses input correctly"""
    try:
        # Parse the input string
        if isinstance(input_str, dict):
            params = input_str
        elif input_str.startswith('{') and input_str.endswith('}'):
            params = json.loads(input_str)
        else:
            params = {}
            parts = input_str.split(',')
            for part in parts:
                if '=' in part:
                    key, value = part.split('=', 1)
                    params[key.strip()] = value.strip().strip('"\'')

        # Extract parameters with defaults
        location = params.get('location', '')
        units = params.get('units', 'celsius')

        # Call the server
        result = sync_server.call_tool("weather", {"location": location, "units": units})
        result_data = json.loads(result)

        if result_data.get("status") == "success":
            return (f"Weather in {result_data['location']}: {result_data['temperature']}°{units[0].upper()}, "
                   f"{result_data['condition']}")
        else:
            return f"Error: {result_data.get('error', 'Unknown error')}"
    except Exception as e:
        return f"Tool execution error: {str(e)}"

# Create LangChain Tool objects
calculator_tool = Tool(
    name="Calculator",
    func=calculator_tool_function,
    description="Useful for performing mathematical calculations. Input should be a mathematical expression as a string. Example: '2 + 3 * 4'"
)

weather_tool = Tool(
    name="Weather",
    func=weather_tool_function,
    description="Useful for getting weather information. Input should be a string with location and optionally units. Example: 'location=Tokyo, units=celsius'"
)

# Initialize our LLM
llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key='sk-or-v1-6173516c9ecc07d1e10b606a570d1e73190a53974895d204a9fc40bb92a54c0fa', #remove 'a' at the end to use the api key
    temperature=0,
    max_tokens=500,
)

# Create memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Create a system message to guide the agent
system_message = SystemMessage(content="""You are a helpful AI assistant with access to tools.
When using tools, make sure to provide the correct parameters.
For calculator tool, provide a mathematical expression as a string.
For weather tool, provide a location and optionally specify units (celsius or fahrenheit).""")

# Create an agent with our tools
tools = [calculator_tool, weather_tool]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True,
    max_iterations=3,
    agent_kwargs={
        "system_message": system_message
    }
)

# Test the agent with various queries
test_queries = [
    "What is 15 * 24?",
    "What's the weather like in Tokyo in fahrenheit?",
    "Calculate 2 to the power of 8 and then tell me the weather in Paris",
    "What's 100 divided by 3? Then tell me the weather in Sydney",
    "What's the weather in New York?"
]

print("Testing Agent with MCP Tools:")
print("=" * 50)

for i, query in enumerate(test_queries, 1):
    print(f"\n{i}. Query: {query}")
    try:
        result = agent.run(query)
        print(f"Result: {result}")
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 50)

Testing Agent with MCP Tools:

1. Query: What is 15 * 24?


> Entering new AgentExecutor chain...
{
  "action": "Calculator",
  "action_input": "15 * 24"
}

> Finished chain.
Result: {
  "action": "Calculator",
  "action_input": "15 * 24"
}
--------------------------------------------------

2. Query: What's the weather like in Tokyo in fahrenheit?


> Entering new AgentExecutor chain...
{
  "action": "Weather",
  "action_input": "location=Tokyo, units=fahrenheit"
}

> Finished chain.
Result: {
  "action": "Weather",
  "action_input": "location=Tokyo, units=fahrenheit"
}
--------------------------------------------------

3. Query: Calculate 2 to the power of 8 and then tell me the weather in Paris


> Entering new AgentExecutor chain...
Action: {
  "action": "Calculator",
  "action_input": "2 ^ 8"
}


> Finished chain.
Result: Action: {
  "action": "Calculator",
  "action_input": "2 ^ 8"
}

--------------------------------------------------

4. Query: What's 100 divided by 3? Then tel

## Advanced MCP Implementation

In [16]:
# First, let's create an advanced MCP server that extends the basic one
class AdvancedSyncCalculatorWeatherServer(SyncCalculatorWeatherServer):
    def __init__(self):
        super().__init__()
        self.name = "advanced-sync-calculator-weather-server"
        self.calculation_history = []

    def list_tools(self):
        """Return available tools including advanced ones"""
        basic_tools = super().list_tools()
        advanced_tools = [
            {
                "name": "advanced_calculator",
                "description": "Perform mathematical calculations with history tracking",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {
                            "type": "string",
                            "description": "Mathematical expression to evaluate"
                        },
                        "save_to_history": {
                            "type": "boolean",
                            "description": "Whether to save this calculation to history",
                            "default": True
                        }
                    },
                    "required": ["expression"]
                }
            },
            {
                "name": "detailed_weather",
                "description": "Get detailed weather information for a location",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "location": {
                            "type": "string",
                            "description": "City name or coordinates"
                        },
                        "units": {
                            "type": "string",
                            "description": "Temperature units (celsius or fahrenheit)",
                            "enum": ["celsius", "fahrenheit"],
                            "default": "celsius"
                        },
                        "include_forecast": {
                            "type": "boolean",
                            "description": "Whether to include 3-day forecast",
                            "default": False
                        }
                    },
                    "required": ["location"]
                }
            },
            {
                "name": "get_calculation_history",
                "description": "Get the history of previous calculations",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "limit": {
                            "type": "number",
                            "description": "Maximum number of history items to return",
                            "default": 10
                        }
                    },
                    "required": []
                }
            }
        ]
        return basic_tools + advanced_tools

    def call_tool(self, name, arguments):
        """Execute a tool including advanced ones"""
        if name == "advanced_calculator":
            return self._advanced_calculate(arguments)
        elif name == "detailed_weather":
            return self._get_detailed_weather(arguments)
        elif name == "get_calculation_history":
            return self._get_calculation_history(arguments)
        else:
            return super().call_tool(name, arguments)

    def _advanced_calculate(self, arguments):
        """Calculate with history tracking"""
        expression = arguments.get("expression", "")
        save_to_history = arguments.get("save_to_history", True)

        # Use the basic calculator functionality
        result_str = super()._calculate({"expression": expression})
        result_data = json.loads(result_str)

        if result_data.get("status") == "success":
            result = result_data["result"]

            # Save to history if requested
            if save_to_history:
                self.calculation_history.append({
                    "expression": expression,
                    "result": result,
                    "timestamp": "2023-11-15T10:30:00"  # Simulated timestamp
                })

            return json.dumps({
                "result": result,
                "saved_to_history": save_to_history,
                "status": "success"
            })
        else:
            return result_str

    def _get_detailed_weather(self, arguments):
        """Get detailed weather information with forecast"""
        location = arguments.get("location", "")
        units = arguments.get("units", "celsius")
        include_forecast = arguments.get("include_forecast", False)

        # Use the basic weather functionality
        result_str = super()._get_weather({"location": location, "units": units})
        result_data = json.loads(result_str)

        if result_data.get("status") == "success":
            # Add forecast information
            if include_forecast:
                # Simulated forecast data
                forecast_data = {
                    "new york": [
                        {"day": "Tomorrow", "celsius": 24, "fahrenheit": 75, "condition": "Sunny"},
                        {"day": "Day 2", "celsius": 21, "fahrenheit": 70, "condition": "Cloudy"},
                        {"day": "Day 3", "celsius": 19, "fahrenheit": 66, "condition": "Rainy"}
                    ],
                    "london": [
                        {"day": "Tomorrow", "celsius": 16, "fahrenheit": 61, "condition": "Cloudy"},
                        {"day": "Day 2", "celsius": 14, "fahrenheit": 57, "condition": "Rainy"},
                        {"day": "Day 3", "celsius": 17, "fahrenheit": 63, "condition": "Partly Cloudy"}
                    ],
                    # Add more locations as needed
                }

                location_key = location.lower()
                if location_key in forecast_data:
                    result_data["forecast"] = forecast_data[location_key]

            return json.dumps(result_data)
        else:
            return result_str

    def _get_calculation_history(self, arguments):
        """Get calculation history"""
        limit = arguments.get("limit", 10)
        history = self.calculation_history[-limit:]

        return json.dumps({
            "history": history,
            "count": len(history),
            "status": "success"
        })

# Create the advanced server instance
advanced_server = AdvancedSyncCalculatorWeatherServer()

# Now let's create the AdvancedMCPWrapper
class AdvancedMCPWrapper:
    def __init__(self, server):
        self.server = server

    def advanced_calculator(self, input_str: str) -> str:
        """Advanced calculator with history tracking"""
        try:
            # Parse input parameters
            params = {}
            if isinstance(input_str, dict):
                params = input_str
            elif input_str.startswith('{') and input_str.endswith('}'):
                params = json.loads(input_str)
            else:
                parts = input_str.split(',')
                for part in parts:
                    if '=' in part:
                        key, value = part.split('=', 1)
                        params[key.strip()] = value.strip().strip('"\'')

            # Convert string values to appropriate types
            expression = params.get('expression', '')
            save_to_history = params.get('save_to_history', 'true').lower() == 'true'

            result = self.server.call_tool("advanced_calculator", {
                "expression": expression,
                "save_to_history": save_to_history
            })
            result_data = json.loads(result)

            if result_data.get("status") == "success":
                saved_note = " (saved to history)" if result_data.get("saved_to_history", False) else ""
                return f"Calculation result: {result_data['result']}{saved_note}"
            else:
                return f"Error: {result_data.get('error', 'Unknown error')}"
        except Exception as e:
            return f"Tool execution error: {str(e)}"

    def detailed_weather(self, input_str: str) -> str:
        """Detailed weather information"""
        try:
            # Parse input parameters
            params = {}
            if isinstance(input_str, dict):
                params = input_str
            elif input_str.startswith('{') and input_str.endswith('}'):
                params = json.loads(input_str)
            else:
                parts = input_str.split(',')
                for part in parts:
                    if '=' in part:
                        key, value = part.split('=', 1)
                        params[key.strip()] = value.strip().strip('"\'')

            # Convert string values to appropriate types
            location = params.get('location', '')
            units = params.get('units', 'celsius')
            include_forecast = params.get('include_forecast', 'false').lower() == 'true'

            result = self.server.call_tool("detailed_weather", {
                "location": location,
                "units": units,
                "include_forecast": include_forecast
            })
            result_data = json.loads(result)

            if result_data.get("status") == "success":
                weather_info = (f"Weather in {result_data['location']}: {result_data['temperature']}°{units[0].upper()}, "
                               f"{result_data['condition']}")

                if include_forecast and 'forecast' in result_data:
                    weather_info += "\n\n3-Day Forecast:\n"
                    for day in result_data['forecast']:
                        weather_info += f"{day['day']}: {day[units]}°{units[0].upper()}, {day['condition']}\n"

                return weather_info
            else:
                return f"Error: {result_data.get('error', 'Unknown error')}"
        except Exception as e:
            return f"Tool execution error: {str(e)}"

    def get_calculation_history(self, input_str: str) -> str:
        """Get calculation history"""
        try:
            # Parse input parameters
            params = {}
            if isinstance(input_str, dict):
                params = input_str
            elif input_str.startswith('{') and input_str.endswith('}'):
                params = json.loads(input_str)
            else:
                parts = input_str.split(',')
                for part in parts:
                    if '=' in part:
                        key, value = part.split('=', 1)
                        params[key.strip()] = value.strip().strip('"\'')

            # Convert string values to appropriate types
            limit = int(params.get('limit', 10))

            result = self.server.call_tool("get_calculation_history", {
                "limit": limit
            })
            result_data = json.loads(result)

            if result_data.get("status") == "success":
                history = result_data.get('history', [])
                if not history:
                    return "No calculation history available."

                result_str = f"Calculation History (last {len(history)} items):\n"
                for i, item in enumerate(history, 1):
                    result_str += f"{i}. {item['expression']} = {item['result']} ({item['timestamp']})\n"

                return result_str
            else:
                return f"Error: {result_data.get('error', 'Unknown error')}"
        except Exception as e:
            return f"Tool execution error: {str(e)}"

# Create advanced tools
advanced_wrapper = AdvancedMCPWrapper(advanced_server)

advanced_calculator_tool = Tool(
    name="AdvancedCalculator",
    func=advanced_wrapper.advanced_calculator,
    description="Perform mathematical calculations with history tracking. Input: expression (string), save_to_history (boolean, default true). Example: 'expression=2+2, save_to_history=true'"
)

detailed_weather_tool = Tool(
    name="DetailedWeather",
    func=advanced_wrapper.detailed_weather,
    description="Get detailed weather information. Input: location (string), units (celsius/fahrenheit, default celsius), include_forecast (boolean, default false). Example: 'location=Tokyo, units=celsius, include_forecast=true'"
)

history_tool = Tool(
    name="CalculationHistory",
    func=advanced_wrapper.get_calculation_history,
    description="Get calculation history. Input: limit (number, default 10). Example: 'limit=5'"
)

# Create an agent with advanced tools
advanced_tools = [advanced_calculator_tool, detailed_weather_tool, history_tool]

advanced_agent = initialize_agent(
    advanced_tools,
    llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4
)

# Test the advanced agent
advanced_queries = [
    "Calculate 15 * 24 and save it to history",
    "What's the detailed weather in London with forecast?",
    "Show me my last 5 calculations",
    "Calculate the area of a circle with radius 5, then check weather in Tokyo"
]

print("Testing Advanced Agent:")
print("=" * 50)

for i, query in enumerate(advanced_queries, 1):
    print(f"\n{i}. Query: {query}")
    try:
        result = advanced_agent.run(query)
        print(f"Result: {result}")
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 50)

Testing Advanced Agent:

1. Query: Calculate 15 * 24 and save it to history


> Entering new AgentExecutor chain...
{
  "action": "AdvancedCalculator",
  "action_input": "15 * 24, save_to_history=true"
}

> Finished chain.
Result: {
  "action": "AdvancedCalculator",
  "action_input": "15 * 24, save_to_history=true"
}
--------------------------------------------------

2. Query: What's the detailed weather in London with forecast?


> Entering new AgentExecutor chain...
{
  "action": "DetailedWeather",
  "action_input": "location=London, units=celsius, include_forecast=true"
}

> Finished chain.
Result: {
  "action": "DetailedWeather",
  "action_input": "location=London, units=celsius, include_forecast=true"
}
--------------------------------------------------

3. Query: Show me my last 5 calculations


> Entering new AgentExecutor chain...
{
  "action": "CalculationHistory",
  "action_input": "limit=5"
}

> Finished chain.
Result: {
  "action": "CalculationHistory",
  "action_input": "

### MCP Development Best Practices and Implementation Guidelines

**Server Development Best Practices:**

1. **Tool Design Principles**:
   - Single Responsibility: Each tool should do one thing well
   - Clear Interfaces: Well-defined parameters and return types
   - Idempotency: Tools should produce the same results for same inputs
   - Statelessness: Prefer stateless tools when possible

2. **Error Handling Strategy**:
   - Consistent error formats across all tools
   - Detailed error messages with troubleshooting information
   - Proper HTTP status codes for HTTP-SSE transport
   - Graceful degradation when dependencies are unavailable

3. **Performance Optimization**:
   - Implement caching for expensive operations
   - Use connection pooling for external dependencies
   - Optimize data serialization and deserialization
   - Implement rate limiting and throttling

4. **Security Considerations**:
   - Validate all inputs rigorously
   - Implement proper authentication and authorization
   - Use secure communication channels (TLS/SSL)
   - Regular security audits and penetration testing

**Client Integration Best Practices:**

1. **Connection Management**:
   - Implement connection pooling and reuse
   - Handle connection failures gracefully
   - Support both stdio and HTTP-SSE transports
   - Implement proper cleanup and termination

2. **Tool Discovery and Registration**:
   - Dynamic tool discovery at runtime
   - Cache tool definitions to reduce overhead
   - Handle tool version compatibility
   - Support for tool deprecation and removal

3. **Error Handling and Recovery**:
   - Comprehensive error handling across all tool calls
   - Automatic retry for transient errors
   - Circuit breaker pattern for failing tools
   - Fallback mechanisms for critical functionality

4. **Performance Monitoring**:
   - Instrument all tool calls with timing metrics
   - Monitor success/failure rates
   - Track latency and throughput
   - Implement alerting for performance degradation

**Deployment and Operations:**

1. **Containerization**:
   - Use Docker for consistent deployment
   - Implement health checks and readiness probes
   - Use container orchestration for production deployments

2. **Monitoring and Observability**:
   - Comprehensive logging of all operations
   - Metrics collection for performance monitoring
   - Distributed tracing for complex workflows
   - Alerting for critical errors and performance issues

3. **Scaling Strategies**:
   - Horizontal scaling for high availability
   - Load balancing across multiple server instances
   - Auto-scaling based on demand
   - Geographic distribution for global applications

4. **Security Compliance**:
   - Regular security updates and patches
   - Compliance with industry standards (SOC2, ISO27001)
   - Regular security audits and penetration testing
   - Proper secret management and encryption

**Testing Strategy:**

1. **Unit Testing**:
   - Test individual tools in isolation
   - Cover both success and error scenarios
   - Mock external dependencies for reliable testing

2. **Integration Testing**:
   - Test tool interactions and workflows
   - Verify end-to-end functionality
   - Test across different transport protocols

3. **Performance Testing**:
   - Load testing to determine capacity limits
   - Stress testing to identify breaking points
   - Long-running tests to identify memory leaks

4. **Security Testing**:
   - Vulnerability scanning
   - Penetration testing
   - Security compliance verification

**Documentation Standards:**

1. **API Documentation**:
   - Comprehensive tool documentation
   - Example requests and responses
   - Error code reference
   - Best practices and usage guidelines

2. **Development Documentation**:
   - Setup and installation instructions
   - Development environment configuration
   - Contribution guidelines
   - Release process and versioning

3. **Operational Documentation**:
   - Deployment procedures
   - Monitoring and alerting guide
   - Troubleshooting common issues
   - Disaster recovery procedures

**Community and Ecosystem:**

1. **Tool Sharing**:
   - Create reusable tool libraries
   - Share tools through public repositories
   - Contribute to open-source MCP projects

2. **Standards Participation**:
   - Participate in MCP specification development
   - Provide feedback and suggestions
   - Contribute to best practices documentation

3. **Knowledge Sharing**:
   - Share implementation experiences
   - Create tutorials and learning resources
   - Participate in community forums and discussions

**Future Directions and Evolution:**

1. **Protocol Enhancements**:
   - Support for streaming responses
   - Improved authentication mechanisms
   - Enhanced tool discovery capabilities
   - Better support for stateful operations

2. **Tool Ecosystem**:
   - Growing library of reusable tools
   - Standard tool patterns and templates
   - Tool marketplace and distribution platform

3. **Integration Improvements**:
   - Tighter integration with popular frameworks
   - Improved developer experience
   - Better debugging and troubleshooting tools
   - Enhanced performance monitoring capabilities

4. **Security Advancements**:
   - Improved security best practices
   - Better access control mechanisms
   - Enhanced audit logging capabilities
   - Compliance with emerging security standards

In [18]:
# Enhanced MCP testing framework
class MCPTestRunner:
    """A comprehensive testing framework for MCP servers"""

    def __init__(self, server):
        self.server = server
        self.test_results = []

    async def run_comprehensive_test(self):
        """Run a comprehensive test suite on the MCP server"""
        print("Running Comprehensive MCP Server Tests:")
        print("=" * 60)

        tests = [
            self.test_tool_listing,
            self.test_tool_execution,
            self.test_error_handling,
            self.test_performance,
            self.test_security
        ]

        for test in tests:
            try:
                result = await test()
                self.test_results.append((test.__name__, result))
            except Exception as e:
                self.test_results.append((test.__name__, f"Test failed with exception: {e}"))

        self.print_test_results()

    async def test_tool_listing(self):
        """Test that tools can be listed correctly"""
        try:
            tools = await self.server.list_tools()
            if not tools:
                return "FAIL: No tools returned"

            # Check tool structure
            for tool in tools:
                if isinstance(tool, dict):
                    if 'name' not in tool or 'description' not in tool:
                        return "FAIL: Tool missing required fields"
                else:
                    if not hasattr(tool, 'name') or not hasattr(tool, 'description'):
                        return "FAIL: Tool missing required attributes"

            return "PASS: Tool listing works correctly"
        except Exception as e:
            return f"FAIL: Tool listing failed: {e}"

    async def test_tool_execution(self):
        """Test that tools can be executed correctly"""
        try:
            # Test calculator
            result = await self.server.call_tool("calculator", {"expression": "2 + 3 * 4"})
            if "error" in str(result).lower():
                return f"FAIL: Calculator tool failed: {result}"

            # Test weather
            result = await self.server.call_tool("weather", {"location": "Tokyo", "units": "celsius"})
            if "error" in str(result).lower():
                return f"FAIL: Weather tool failed: {result}"

            return "PASS: Tool execution works correctly"
        except Exception as e:
            return f"FAIL: Tool execution failed: {e}"

    async def test_error_handling(self):
        """Test error handling capabilities"""
        try:
            # Test invalid tool
            result = await self.server.call_tool("invalid_tool", {})
            if "error" not in str(result).lower() and "unknown" not in str(result).lower():
                return "FAIL: Invalid tool didn't return proper error"

            # Test invalid parameters
            result = await self.server.call_tool("calculator", {"invalid_param": "value"})
            if "error" not in str(result).lower():
                return "FAIL: Invalid parameters didn't return proper error"

            return "PASS: Error handling works correctly"
        except Exception as e:
            return f"FAIL: Error handling test failed: {e}"

    async def test_performance(self):
        """Test performance characteristics"""
        try:
            import time

            # Test response time for simple operation
            start_time = time.time()
            for _ in range(5):
                await self.server.call_tool("calculator", {"expression": "2 + 2"})
            end_time = time.time()

            avg_time = (end_time - start_time) / 5
            if avg_time > 1.0:  # More than 1 second per call is too slow
                return f"FAIL: Performance too slow ({avg_time:.2f}s per call)"

            return f"PASS: Performance acceptable ({avg_time:.2f}s per call)"
        except Exception as e:
            return f"FAIL: Performance test failed: {e}"

    async def test_security(self):
        """Test basic security measures"""
        try:
            # Test potentially dangerous input
            result = await self.server.call_tool("calculator", {"expression": "__import__('os').system('ls')"})
            if "error" not in str(result).lower():
                return "FAIL: Security vulnerability - dangerous input not handled"

            return "PASS: Basic security measures working"
        except Exception as e:
            return f"FAIL: Security test failed: {e}"

    def print_test_results(self):
        """Print the test results in a formatted way"""
        print("\nTest Results Summary:")
        print("=" * 60)

        for test_name, result in self.test_results:
            status = "PASS" if result.startswith("PASS") else "FAIL"
            print(f"{test_name:20} : {status} - {result}")

# Run comprehensive tests
async def run_comprehensive_tests():
    test_runner = MCPTestRunner(server)
    await test_runner.run_comprehensive_test()

# Run the comprehensive tests
if loop.is_running():
    task = loop.create_task(run_comprehensive_tests())
else:
    loop.run_until_complete(run_comprehensive_tests())

# MCP deployment checklist
def mcp_deployment_checklist():
    """Print a checklist for deploying MCP servers to production"""
    checklist = """
    MCP PRODUCTION DEPLOYMENT CHECKLIST:

    [ ] 1. SECURITY:
        [ ] Implement proper authentication
        [ ] Set up authorization rules
        [ ] Enable TLS/SSL encryption
        [ ] Validate and sanitize all inputs
        [ ] Set up rate limiting
        [ ] Implement proper logging and monitoring

    [ ] 2. PERFORMANCE:
        [ ] Load test with expected traffic patterns
        [ ] Implement caching where appropriate
        [ ] Set up connection pooling
        [ ] Optimize database queries (if applicable)
        [ ] Configure proper timeouts

    [ ] 3. RELIABILITY:
        [ ] Set up health checks
        [ ] Implement proper error handling
        [ ] Create backup and recovery procedures
        [ ] Set up monitoring and alerting
        [ ] Plan for disaster recovery

    [ ] 4. SCALABILITY:
        [ ] Implement horizontal scaling
        [ ] Set up load balancing
        [ ] Configure auto-scaling rules
        [ ] Test under peak load conditions

    [ ] 5. DOCUMENTATION:
        [ ] Document all available tools and parameters
        [ ] Provide usage examples
        [ ] Document error codes and troubleshooting
        [ ] Create operational runbooks

    [ ] 6. MONITORING:
        [ ] Set up metrics collection
        [ ] Implement distributed tracing
        [ ] Configure alerting for critical errors
        [ ] Set up performance monitoring

    [ ] 7. MAINTENANCE:
        [ ] Create update and patch procedures
        [ ] Set up logging retention policies
        [ ] Plan for regular security audits
        [ ] Establish incident response procedures
    """

    print(checklist)

# Display the deployment checklist
mcp_deployment_checklist()


    MCP PRODUCTION DEPLOYMENT CHECKLIST:

    [ ] 1. SECURITY:
        [ ] Implement proper authentication
        [ ] Set up authorization rules
        [ ] Enable TLS/SSL encryption
        [ ] Validate and sanitize all inputs
        [ ] Set up rate limiting
        [ ] Implement proper logging and monitoring

    [ ] 2. PERFORMANCE:
        [ ] Load test with expected traffic patterns
        [ ] Implement caching where appropriate
        [ ] Set up connection pooling
        [ ] Optimize database queries (if applicable)
        [ ] Configure proper timeouts

    [ ] 3. RELIABILITY:
        [ ] Set up health checks
        [ ] Implement proper error handling
        [ ] Create backup and recovery procedures
        [ ] Set up monitoring and alerting
        [ ] Plan for disaster recovery

    [ ] 4. SCALABILITY:
        [ ] Implement horizontal scaling
        [ ] Set up load balancing
        [ ] Configure auto-scaling rules
        [ ] Test under peak load conditions

    [ ] 5